# Experiment 6 :: GloVe Embeddings and N-gram Language Model

- 6.1 GloVe word embeddings, most similar words of "king"
  - a. without any pretrained model
  - b. using a pretrained model (Gensim)
- 6.2 N-gram language model from scratch to predict the next word

In [1]:
%pip install -q numpy gensim

Note: you may need to restart the kernel to use updated packages.


## 6.1 a. GloVe without any pretrained model

In [2]:
import re
import numpy as np

text = open('glove_corpus.txt', encoding='utf-8').read()
sentences = [line.strip() for line in text.splitlines() if line.strip()]

stop_words = {'the', 'a', 'is', 'and', 'in', 'of', 'at', 'to'}

tokenized_sentences = []
for sentence in sentences:
    words = re.findall(r'[a-z]+', sentence.lower())
    words = [w for w in words if w not in stop_words]
    tokenized_sentences.append(words)

vocab = sorted(set(w for words in tokenized_sentences for w in words))
word_to_id = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print('Sentences:', len(sentences))
print('Vocabulary size:', vocab_size)
print(tokenized_sentences[:3])

Sentences: 24
Vocabulary size: 61
[['king', 'rules', 'kingdom'], ['queen', 'rules', 'kingdom'], ['king', 'powerful', 'man']]


In [3]:
window_size = 3
cooccurrence = np.zeros((vocab_size, vocab_size))

for words in tokenized_sentences:
    for i, word in enumerate(words):
        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if i != j:
                distance = abs(i - j)
                cooccurrence[word_to_id[word], word_to_id[words[j]]] += 1 / distance

print('Non zero pairs:', np.count_nonzero(cooccurrence))
print('king row:', {vocab[j]: round(float(cooccurrence[word_to_id['king'], j]), 2) for j in np.nonzero(cooccurrence[word_to_id['king']])[0]})

Non zero pairs: 220
king row: {'kingdom': 0.5, 'lives': 1.0, 'man': 0.5, 'palace': 0.33, 'powerful': 1.0, 'prince': 0.5, 'queen': 1.0, 'royal': 0.5, 'rule': 0.5, 'rules': 1.0, 'son': 1.0, 'together': 0.33}


In [4]:
np.random.seed(42)

embedding_dim = 20
x_max = 10
alpha = 0.75
learning_rate = 0.05
epochs = 1000

W = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
W_context = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
b = np.zeros(vocab_size)
b_context = np.zeros(vocab_size)

grad_sq_W = np.ones_like(W)
grad_sq_W_context = np.ones_like(W_context)
grad_sq_b = np.ones_like(b)
grad_sq_b_context = np.ones_like(b_context)

pairs = list(zip(*np.nonzero(cooccurrence)))

for epoch in range(epochs):
    total_loss = 0
    np.random.shuffle(pairs)
    for i, j in pairs:
        x_ij = cooccurrence[i, j]
        weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1

        diff = W[i] @ W_context[j] + b[i] + b_context[j] - np.log(x_ij)
        total_loss += 0.5 * weight * diff ** 2

        grad = weight * diff
        grad_W = grad * W_context[j]
        grad_W_context = grad * W[i]

        W[i] -= learning_rate * grad_W / np.sqrt(grad_sq_W[i])
        W_context[j] -= learning_rate * grad_W_context / np.sqrt(grad_sq_W_context[j])
        b[i] -= learning_rate * grad / np.sqrt(grad_sq_b[i])
        b_context[j] -= learning_rate * grad / np.sqrt(grad_sq_b_context[j])

        grad_sq_W[i] += grad_W ** 2
        grad_sq_W_context[j] += grad_W_context ** 2
        grad_sq_b[i] += grad ** 2
        grad_sq_b_context[j] += grad ** 2

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch:4d}  loss {total_loss:.4f}')

Epoch    0  loss 3.5358


Epoch  100  loss 1.5526


Epoch  200  loss 1.3066


Epoch  300  loss 0.8967


Epoch  400  loss 0.5038


Epoch  500  loss 0.1826
Epoch  600  loss 0.0556


Epoch  700  loss 0.0219
Epoch  800  loss 0.0112


Epoch  900  loss 0.0066
Epoch  999  loss 0.0043


In [5]:
embeddings = W + W_context

def most_similar(word, top_n=5):
    vector = embeddings[word_to_id[word]]
    scores = embeddings @ vector / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(vector))
    ranked = np.argsort(-scores)
    return [(vocab[i], round(float(scores[i]), 4)) for i in ranked if vocab[i] != word][:top_n]

print('Most similar to "king":')
for word, score in most_similar('king'):
    print(f'  {word:<12}{score}')

Most similar to "king":
  son         0.9813
  queen       0.966
  wise        0.9437
  powerful    0.915
  prince      0.8279


In [6]:
for word in ['queen', 'man', 'cat', 'computer']:
    print(word, '->', most_similar(word, 3))

queen -> [('king', 0.966), ('son', 0.9424), ('wise', 0.9274)]
man -> [('woman', 0.9089), ('strong', 0.6145), ('cat', 0.49)]
cat -> [('dog', 0.9596), ('drinks', 0.5341), ('man', 0.49)]
computer -> [('processes', 0.9831), ('scientist', 0.8801), ('conducts', 0.7779)]


## 6.1 b. GloVe using pretrained model (Gensim)

In [7]:
import gensim.downloader as api

glove_model = api.load('glove-wiki-gigaword-100')

print('Vocabulary size:', len(glove_model.key_to_index))
print('Vector size:', glove_model.vector_size)

Vocabulary size: 400000
Vector size: 100


In [8]:
print('Most similar to "king":')
for word, score in glove_model.most_similar('king', topn=10):
    print(f'  {word:<12}{score:.4f}')

Most similar to "king":
  prince      0.7682
  queen       0.7508
  son         0.7021
  brother     0.6986
  monarch     0.6978
  throne      0.6920
  kingdom     0.6811
  father      0.6802
  emperor     0.6713
  ii          0.6676


In [9]:
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print('king - man + woman =', result)

print('similarity(king, queen) =', round(float(glove_model.similarity('king', 'queen')), 4))
print('similarity(king, cat)   =', round(float(glove_model.similarity('king', 'cat')), 4))

king - man + woman = [('queen', 0.7698541283607483), ('monarch', 0.6843381524085999), ('throne', 0.6755736470222473)]
similarity(king, queen) = 0.7508
similarity(king, cat)   = 0.3282


## 6.2 N-gram language model

In [10]:
corpus = '''natural language processing is a branch of artificial intelligence. natural language processing enables computers to understand human language. machine learning is a branch of artificial intelligence. deep learning is a part of machine learning. language models learn patterns from text data. a language model predicts the next word in a sentence. natural language processing uses language models. artificial intelligence includes machine learning and deep learning.'''

lm_sentences = [s.strip().split() for s in corpus.split('.') if s.strip()]
lm_sentences = [['<s>', '<s>'] + words + ['</s>'] for words in lm_sentences]

for words in lm_sentences[:3]:
    print(words)

['<s>', '<s>', 'natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', '</s>']
['<s>', '<s>', 'natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language', '</s>']
['<s>', '<s>', 'machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', '</s>']


In [11]:
from collections import Counter

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for words in lm_sentences:
    unigram_counts.update(words)
    bigram_counts.update(zip(words, words[1:]))
    trigram_counts.update(zip(words, words[1:], words[2:]))

lm_vocab = [w for w in unigram_counts if w != '<s>']
V = len(lm_vocab)

print('Vocabulary size:', V)
print('Unique bigrams:', len(bigram_counts))
print('Unique trigrams:', len(trigram_counts))
print('Top bigrams:', bigram_counts.most_common(5))
print('Top trigrams:', trigram_counts.most_common(5))

Vocabulary size: 35
Unique bigrams: 54
Unique trigrams: 61
Top bigrams: [(('<s>', '<s>'), 8), (('<s>', 'natural'), 3), (('natural', 'language'), 3), (('language', 'processing'), 3), (('is', 'a'), 3)]
Top trigrams: [(('<s>', '<s>', 'natural'), 3), (('<s>', 'natural', 'language'), 3), (('natural', 'language', 'processing'), 3), (('is', 'a', 'branch'), 2), (('a', 'branch', 'of'), 2)]


In [12]:
def bigram_prob(prev_word, word):
    return (bigram_counts[(prev_word, word)] + 1) / (unigram_counts[prev_word] + V)

def trigram_prob(w1, w2, word):
    return (trigram_counts[(w1, w2, word)] + 1) / (bigram_counts[(w1, w2)] + V)

print('P(language | natural) =', round(bigram_prob('natural', 'language'), 4))
print('P(processing | natural language) =', round(trigram_prob('natural', 'language', 'processing'), 4))
print('P(learning | natural language) =', round(trigram_prob('natural', 'language', 'learning'), 4))

P(language | natural) = 0.1053
P(processing | natural language) = 0.1053
P(learning | natural language) = 0.0263


In [13]:
def predict_next_word(text, top_n=3):
    words = ['<s>', '<s>'] + text.lower().split()
    w1, w2 = words[-2], words[-1]

    if bigram_counts[(w1, w2)] > 0:
        scores = {w: trigram_prob(w1, w2, w) for w in lm_vocab}
        model_used = 'trigram'
    else:
        scores = {w: bigram_prob(w2, w) for w in lm_vocab}
        model_used = 'bigram'

    best = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
    return model_used, [(w, round(p, 4)) for w, p in best]

test_inputs = ['natural language', 'machine', 'deep learning is a', 'artificial intelligence', 'a language model predicts the', 'language models learn']

for test in test_inputs:
    model_used, predictions = predict_next_word(test)
    print(f'{test!r:<35} [{model_used}] -> {predictions}')

'natural language'                  [trigram] -> [('processing', 0.1053), ('natural', 0.0263), ('language', 0.0263)]
'machine'                           [trigram] -> [('learning', 0.0556), ('natural', 0.0278), ('language', 0.0278)]
'deep learning is a'                [trigram] -> [('branch', 0.0789), ('part', 0.0526), ('natural', 0.0263)]
'artificial intelligence'           [trigram] -> [('</s>', 0.0789), ('includes', 0.0526), ('natural', 0.0263)]
'a language model predicts the'     [trigram] -> [('next', 0.0556), ('natural', 0.0278), ('language', 0.0278)]
'language models learn'             [trigram] -> [('patterns', 0.0556), ('natural', 0.0278), ('language', 0.0278)]


In [14]:
def generate_sentence(start, max_words=12):
    words = start.lower().split()
    for _ in range(max_words):
        _, predictions = predict_next_word(' '.join(words), top_n=1)
        next_word = predictions[0][0]
        if next_word == '</s>':
            break
        words.append(next_word)
    return ' '.join(words)

for start in ['natural', 'deep learning', 'language models']:
    print(generate_sentence(start))

natural language processing is a branch of artificial intelligence
deep learning is a branch of artificial intelligence
language models


In [15]:
import math

def perplexity(sentence):
    words = ['<s>', '<s>'] + sentence.lower().split() + ['</s>']
    log_prob = 0
    for i in range(2, len(words)):
        log_prob += math.log(trigram_prob(words[i - 2], words[i - 1], words[i]))
    return math.exp(-log_prob / (len(words) - 2))

print('Perplexity (seen):  ', round(perplexity('machine learning is a branch of artificial intelligence'), 2))
print('Perplexity (unseen):', round(perplexity('computers learn language from data'), 2))

Perplexity (seen):   14.44
Perplexity (unseen): 36.22
